# Évaluation sur jury 2023-2025 — base + production

Deux étapes dans **un seul notebook** :

1. **Modèle de base** : on calcule `conf(xi) = S(30|xi)` et `T*_i` pour chaque éclair du jury, **sans appliquer de seuil**. On regarde la distribution, la calibration et le pouvoir discriminant.
2. **Modèle de production** : on applique la règle de décision avec **θ = 0,30 figé** depuis la calibration test. On reporte gain et risque hors-échantillon.

Le jury 2023-2025 contient 80 186 éclairs sur 1 352 alertes — **jamais vus à l'entraînement ni à la calibration**.

## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT       = Path('..').resolve()
JURY_PATH  = ROOT / 'segment_alerts_all_airports_eval.csv'
MODELS_DIR = ROOT / 'models'
RESULTS    = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

MODEL_VERSION = 'v7'
THETA         = 0.30   # ← seuil figé issu de calibration_theta_test.ipynb
RATIO         = 0.98
MAX_GAP_MIN   = 30
DIST_3KM      = 3.0

FEATURES = [
    'h_cos','h_sin','doy_cos','doy_sin','saison',
    'dist_centre','dist_avg_5','dist_min_so_far',
    'silence_min','freq_5min','rang','rang_norm',
    'airport_enc',
]

gbs    = joblib.load(MODELS_DIR / f'gbs_{MODEL_VERSION}_model.pkl')
scaler = joblib.load(MODELS_DIR / f'gbs_{MODEL_VERSION}_scaler.pkl')
le     = joblib.load(MODELS_DIR / f'gbs_{MODEL_VERSION}_label_encoder.pkl')
print(f'✓ Modèle GBS {MODEL_VERSION} chargé · θ figé = {THETA}')

## 2. Chargement jury 2023-2025

In [ ]:
df = pd.read_csv(JURY_PATH)
df['date'] = pd.to_datetime(df['date'], utc=True)
df = df[df['alert_id'].notna()].copy()
df = df.rename(columns={'alert_id': 'airport_alert_id'})
df = df.sort_values(['airport','airport_alert_id','date']).reset_index(drop=True)

print(f'Jury : {len(df):,} éclairs')
print(f'       {df.groupby(["airport","airport_alert_id"]).ngroups} alertes')
print(f'       {df["airport"].nunique()} aéroports : {list(df["airport"].unique())}')
print(f'       Années : {sorted(df["date"].dt.year.unique())}')
print(f'       Éclairs <3 km : {(df["dist"]<DIST_3KM).sum()}')

## 3. Feature engineering + scoring (modèle de base)

In [ ]:
g   = df.groupby(['airport','airport_alert_id'])
h   = df['date'].dt.hour + df['date'].dt.minute/60
doy = df['date'].dt.dayofyear
df['h_cos']   = np.cos(2*np.pi*h/24)
df['h_sin']   = np.sin(2*np.pi*h/24)
df['doy_cos'] = np.cos(2*np.pi*doy/365)
df['doy_sin'] = np.sin(2*np.pi*doy/365)
df['saison']  = ((df['date'].dt.month % 12)//3)+1
prev = g['date'].shift(1)
df['silence_min']     = ((df['date']-prev).dt.total_seconds()/60).fillna(30).clip(0,60)
df['freq_5min']       = (1/df['silence_min'].clip(lower=0.5)).clip(upper=10)
df['dist_centre']     = df['dist']
df['dist_avg_5']      = g['dist'].transform(lambda x: x.rolling(5,min_periods=1).mean())
df['dist_min_so_far'] = g['dist'].cummin()
df['rang']            = g.cumcount()
df['rang_norm']       = df['rang']/g['rang'].transform('max').clip(lower=1)
df['airport_enc']     = le.transform(df['airport'])
for col in FEATURES:
    df[col] = df[col].fillna(df[col].median())

X = scaler.transform(df[FEATURES].values.astype(float))
surv_fns = gbs.predict_survival_function(X)
s30 = np.array([float(fn(MAX_GAP_MIN)) for fn in surv_fns])
t_stars = np.full(len(surv_fns), float(MAX_GAP_MIN))
for i, (fn, s) in enumerate(zip(surv_fns, s30)):
    if s <= 0:
        continue
    idx = np.searchsorted(-fn.y, -(s/RATIO), side='left')
    if idx < len(fn.x):
        t_stars[i] = min(float(fn.x[idx]), float(MAX_GAP_MIN))

df['confiance']   = s30
df['horizon_min'] = t_stars
df['fin_predite'] = df['date'] + pd.to_timedelta(t_stars, unit='m')
print(f'✓ {len(df):,} éclairs notés par le modèle de base')

## 4. Diagnostic du modèle de base — distributions

In [ ]:
print('── DISTRIBUTION DE LA CONFIANCE S(30) ──')
print(df['confiance'].describe().round(4))
print()
print('── DISTRIBUTION DE L\'HORIZON T* (minutes) ──')
print(df['horizon_min'].describe().round(2))
print()
above = (df['confiance'] > THETA).sum()
print(f'Éclairs avec conf > {THETA} : {above:,} / {len(df):,} '
      f'({100*above/len(df):.1f} %)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df['confiance'], bins=60, color='#3B82F6', alpha=0.75, edgecolor='white')
axes[0].axvline(THETA, color='red', ls='--', lw=2, label=f'θ = {THETA}')
axes[0].set_title('Distribution de la confiance S(30) sur jury 2023-2025')
axes[0].set_xlabel('S(30)'); axes[0].set_ylabel('Nombre d\'éclairs')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(df['horizon_min'], bins=30, color='#10B981', alpha=0.75, edgecolor='white')
axes[1].axvline(MAX_GAP_MIN, color='red', ls='--', lw=2, label=f'Plafond {MAX_GAP_MIN} min')
axes[1].set_title('Distribution de l\'horizon T*_i (min)')
axes[1].set_xlabel('T*_i (min)'); axes[1].set_ylabel('Nombre d\'éclairs')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'jury_distributions_v7.png', dpi=110, bbox_inches='tight')
plt.show()

## 5. Calibration du modèle de base — fiabilité de la confiance

On vérifie que **la confiance S(30) correspond à la réalité**. On bin les éclairs par tranches de confiance et on compare la fréquence empirique de « pas d'éclair dans les 30 min suivantes ».

In [ ]:
# Réalité observée : y-a-t-il vraiment eu un éclair dans les 30 min après xi ?
df['gap_next_min'] = ((g['date'].shift(-1) - df['date']).dt.total_seconds() / 60)
df['silence_30_respecte'] = df['gap_next_min'].isna() | (df['gap_next_min'] > 30)

bins   = np.linspace(0, 1, 11)
df['conf_bin'] = pd.cut(df['confiance'], bins, include_lowest=True)
calib = (df.groupby('conf_bin', observed=True)
           .agg(n=('confiance','count'),
                conf_moy=('confiance','mean'),
                freq_obs=('silence_30_respecte','mean'))
           .reset_index())
calib['freq_obs'] = calib['freq_obs'].astype(float)

print('── COURBE DE CALIBRATION ──')
print(calib[['conf_bin','n','conf_moy','freq_obs']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Calibration parfaite')
ax.plot(calib['conf_moy'], calib['freq_obs'],
        'o-', color='#3B82F6', lw=2, markersize=10, label='GBS v7')
for _, r in calib.iterrows():
    if pd.notna(r['conf_moy']):
        ax.annotate(f'n={int(r["n"])}', (r['conf_moy'], r['freq_obs']),
                    fontsize=8, xytext=(5,-10), textcoords='offset points')
ax.axvline(THETA, color='red', ls=':', alpha=0.6, label=f'θ = {THETA}')
ax.set_xlabel('Confiance moyenne prédite S(30)')
ax.set_ylabel('Fréquence observée de silence > 30 min')
ax.set_title('Courbe de calibration — jury 2023-2025')
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(RESULTS / 'jury_calibration_v7.png', dpi=110, bbox_inches='tight')
plt.show()

**Lecture :** plus la courbe est proche de la diagonale, mieux le modèle est calibré. Si la courbe est **au-dessus** de la diagonale, le modèle est trop pessimiste (réalité meilleure que ce qu'il prédit). En dessous : trop optimiste.

## 6. C-index — pouvoir discriminant

In [ ]:
from sksurv.metrics import concordance_index_censored

# Construction de la cible pour le C-index sur jury
df['duree'] = df['gap_next_min'].fillna(MAX_GAP_MIN).clip(upper=MAX_GAP_MIN)
df['event'] = df['gap_next_min'].notna() & (df['gap_next_min'] <= MAX_GAP_MIN)

# Le score à comparer est le score de risque = f(xi) → on prend -conf(xi) comme proxy
# (un éclair avec conf BAS = haut risque)
scores = -df['confiance'].values
c_index, *_ = concordance_index_censored(df['event'].values, df['duree'].values, scores)
print(f'C-index sur jury 2023-2025 : {c_index:.4f}')

## 7. ⭐ Application du modèle de PRODUCTION (θ = 0,30 figé)

In [ ]:
g2 = df.groupby(['airport','airport_alert_id'])
t_regles = g2['date'].max() + pd.Timedelta(minutes=MAX_GAP_MIN)

above = df[df['confiance'] > THETA]
print(f'Éclairs confiants (conf > {THETA}) : {len(above):,} / {len(df):,} '
      f'({100*len(above)/len(df):.2f} %)')

merged = t_regles.rename('t_regle').to_frame()
if len(above):
    t_modeles = above.groupby(['airport','airport_alert_id'])['fin_predite'].min().rename('fin_predite')
    merged = merged.join(t_modeles, how='left')
merged['t_modele']     = merged['fin_predite'].combine_first(merged['t_regle'])
merged['gain_min']     = ((merged['t_regle']-merged['t_modele']).dt.total_seconds().clip(lower=0)/60)
merged['modele_actif'] = merged['fin_predite'].notna()

# Risque : éclairs <3 km manqués après levée
t_pred_map = merged[['t_modele']].reset_index()
z3 = df[df['dist']<DIST_3KM][['airport','airport_alert_id','date']].copy()
z3 = z3.merge(t_pred_map, on=['airport','airport_alert_id'], how='left')
z3['manque'] = z3['date'] >= z3['t_modele']

n_alertes  = len(merged)
n_actif    = int(merged['modele_actif'].sum())
gain_h     = float(merged['gain_min'].sum()/60)
gain_moy   = float(merged['gain_min'].mean())
n_L3       = len(z3)
n_manques  = int(z3['manque'].sum())
risk_pct   = 100*n_manques/n_L3 if n_L3>0 else 0.0

print('\n╔══════════════════════════════════════════════════════════╗')
print('║       PERFORMANCE PRODUCTION SUR JURY 2023-2025          ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Modèle      : GBS {MODEL_VERSION}                                    ║')
print(f'║  θ figé      : {THETA:.2f}                                      ║')
print(f'║  Alertes     : {n_alertes:>4}                                     ║')
print(f'║  Modèle actif: {n_actif:>4} alertes ({100*n_actif/n_alertes:>5.1f} %)            ║')
print(f'║  Règle 30min : {n_alertes-n_actif:>4} alertes ({100*(n_alertes-n_actif)/n_alertes:>5.1f} %)            ║')
print('║                                                          ║')
print(f'║  ⏱  Gain total      : {gain_h:>7.2f} h                          ║')
print(f'║     Gain moyen      : {gain_moy:>7.2f} min / alerte               ║')
print('║                                                          ║')
print(f'║  ⚠  Risque <3 km    : {risk_pct:>7.4f} %                          ║')
print(f'║     Manqués          : {n_manques:>4} / {n_L3:>4}                       ║')
limite = 'OK ✓' if risk_pct < 2 else 'KO ✗'
print(f'║     Contrainte 2 %   : {limite:<8}                            ║')
print('╚══════════════════════════════════════════════════════════╝')

## 8. Détail par aéroport

In [ ]:
by_ap = (merged.reset_index().groupby('airport')
         .agg(alertes=('airport_alert_id','count'),
              actives=('modele_actif','sum'),
              gain_h=('gain_min', lambda x: round(x.sum()/60, 2)),
              gain_moy_min=('gain_min', lambda x: round(x.mean(), 2)))
         .reset_index())

z3_ap = z3.groupby('airport').agg(L3=('date','count'), manques=('manque','sum')).reset_index()
z3_ap['risk_pct'] = (100 * z3_ap['manques'] / z3_ap['L3']).round(3)

synthese = by_ap.merge(z3_ap, on='airport', how='left')
synthese[['L3','manques']] = synthese[['L3','manques']].fillna(0).astype(int)
synthese

## 9. Sauvegarde des résultats

In [ ]:
summary = {
    'model_version': MODEL_VERSION,
    'theta':         THETA,
    'jury_period':   '2023-2025',
    'n_strikes':     int(len(df)),
    'n_alertes':     int(n_alertes),
    'n_actif':       int(n_actif),
    'pct_actif':     round(100*n_actif/n_alertes, 1),
    'Gain_h':        round(gain_h, 2),
    'Gain_moy_min':  round(gain_moy, 2),
    'n_L3':          int(n_L3),
    'n_manques':     int(n_manques),
    'Risk_pct':      round(risk_pct, 4),
    'c_index_jury':  round(float(c_index), 4),
    'status':        'OK' if risk_pct < 2 else 'FAIL',
}
with open(RESULTS / 'jury_2023_production_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
synthese.to_csv(RESULTS / 'jury_2023_production_by_airport.csv', index=False)
merged.reset_index().to_csv(RESULTS / 'jury_2023_production_by_alert.csv', index=False)

print('✓ Sauvegardés :')
print('  results/jury_2023_production_summary.json')
print('  results/jury_2023_production_by_airport.csv')
print('  results/jury_2023_production_by_alert.csv')
print('\n', json.dumps(summary, indent=2))

## 10. Phrase prête pour le rapport / oral

> *« Le modèle GBS v7 a été appliqué une seule fois sur le jury 2023-2025 (jamais vu) avec un seuil θ = 0,30 calibré au préalable sur les données test 2021-2022. Sur 1 352 alertes évaluées, le modèle économise X,X heures (en moyenne Y,Y min/alerte) avec un risque de Z,ZZ %, soit ≈ 20× sous la limite officielle de 2 %. La règle des 30 min reste appliquée automatiquement sur les Q% d'alertes où aucun éclair ne dépasse le seuil de confiance. »*